In [1]:
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import kagglehub
import os

In [2]:
url = "https://www.kaggle.com/competitions/playground-series-s6e3/data"
test = "test.csv"
train = "train.csv"

In [3]:
# 1. Download the competition dataset
# This will download the files and return the local path where they are cached
dataset_path = kagglehub.competition_download("playground-series-s6e3")
print(f"Data downloaded to: {dataset_path}")

# 2. Load the train and test CSV files into pandas DataFrames
train_df = pd.read_csv(os.path.join(dataset_path, "train.csv"))
test_df = pd.read_csv(os.path.join(dataset_path, "test.csv"))

# Show the first few rows to verify
train_df

# Check the balance of the target variable. Churn 1 is 22.5%
#

Data downloaded to: C:\Users\zak\.cache\kagglehub\competitions\playground-series-s6e3


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,594189,Male,0,No,No,57,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),97.55,5460.70,No
594190,594190,Female,0,No,No,72,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic),91.95,6782.15,No
594191,594191,Female,0,Yes,No,72,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),24.40,1871.90,No
594192,594192,Female,0,No,No,32,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,86.00,2847.20,No


## What are we optimising?

Binary classifier. ROC AUC.



In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import make_colu

In [5]:


pipe = Pipeline(steps=[

])

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

# 1. Load data
print("Loading dataset...")
df = train_df

# 2. Data Cleaning & Preprocessing
# Fix 'TotalCharges' which contains empty strings for 0 tenure customers
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Drop the 'id' column
if 'id' in df.columns:
    df = df.drop('id', axis=1)

# Encode categorical variables
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if col != 'Churn':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

# Encode target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 3. Train/Test Split
X = df.drop('Churn', axis=1)
y = df['Churn']

# Stratify=y ensures the 80/20 split maintains the same proportion of churners
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Train the SOTA Model (HistGradientBoosting)
print("Training model...")
model = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=200,          # Number of boosting iterations/trees
    learning_rate=0.05,    # Shrinks contribution of each tree
    max_leaf_nodes=31      # Controls tree complexity
)
model.fit(X_train, y_train)

# 5. Predictions & Evaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\n--- Model Performance ---")
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_proba), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Loading dataset...


C:\Users\zak\AppData\Local\Temp\ipykernel_4728\2548575162.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


Training model...

--- Model Performance ---
ROC-AUC Score: 0.9153

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.92      0.91     92076
           1       0.71      0.64      0.67     26763

    accuracy                           0.86    118839
   macro avg       0.80      0.78      0.79    118839
weighted avg       0.86      0.86      0.86    118839

